# Lab 14: Motor Adaptation and Subcortical Learning

**Course:** Computational Sensorimotor Control | Week 14 of 15

**Theme:** This lab implements the two computational learning systems from the lecture: **error-based adaptation** (cerebellum → forward model A, B) and **reward-based learning** (basal ganglia → cost function Q, R). You will build the two-stage state-space model (Smith et al., 2006), apply it to force-field adaptation on the nonlinear arm, simulate cerebellar degradation, and implement the REINFORCE algorithm for visuomotor rotation.

**Prerequisites:** Labs 12–13 (OFC/iLQG). The `smc` package (v0.2.0).

**Exercises:** 8 exercises, ~15 lines of student code total.

## Setup

In [ ]:
# Install the smc package (run once)
!pip install --break-system-packages -q git+https://github.com/tarkeshsingh/computational-sensorimotor-control.git#subdirectory=smc

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from smc import Arm, Q_REF, make_hill_muscles

arm = Arm()
rng = np.random.default_rng(42)

plt.rcParams.update({
    'figure.figsize': (12, 4), 'font.size': 11,
    'axes.grid': True, 'grid.alpha': 0.3,
    'axes.spines.top': False, 'axes.spines.right': False,
})
print('Setup complete.')

## Part 1: The Two-Stage State-Space Model

Smith, Ghazizadeh & Shadmehr (2006) proposed that motor adaptation is the sum of two hidden processes with different time constants:

$$x_f(n+1) = A_f \cdot x_f(n) + \eta_f \cdot e(n) \qquad \text{(fast: learns quickly, forgets quickly)}$$

$$x_s(n+1) = A_s \cdot x_s(n) + \eta_s \cdot e(n) \qquad \text{(slow: learns slowly, retains well)}$$

$$x_{\text{total}}(n) = x_f(n) + x_s(n), \qquad e(n) = p(n) - x_{\text{total}}(n)$$

where $p(n)$ is the perturbation on trial $n$, and $e(n)$ is the error.

### Exercise 1: Implement the two-stage update (2 lines)

Complete the three missing lines inside the loop: update `xf`, update `xs`, compute `x_total`.

In [ ]:
def simulate_two_stage(perturbation, Af=0.85, As=0.998, eta_f=0.15, eta_s=0.03):
    """Simulate the two-stage state-space model.
    
    Parameters
    ----------
    perturbation : (N,) array — perturbation schedule (0 during baseline, nonzero during adaptation)
    Af, As : float — retention rates (fast, slow)
    eta_f, eta_s : float — learning rates (fast, slow)
    
    Returns
    -------
    error, xf_hist, xs_hist, xtotal_hist : arrays of shape (N,)
    """
    N = len(perturbation)
    xf = 0.0; xs = 0.0
    error = np.zeros(N); xf_hist = np.zeros(N); xs_hist = np.zeros(N); xtotal_hist = np.zeros(N)
    
    for n in range(N):
        x_total = xf + xs
        e = perturbation[n] - x_total
        error[n] = e
        xf_hist[n] = xf; xs_hist[n] = xs; xtotal_hist[n] = x_total
        
        ### YOUR CODE: update xf, xs (3 lines) ###
        # xf = ...
        # xs = ...
    
    return error, xf_hist, xs_hist, xtotal_hist

### Run it: Baseline → Adaptation → Washout

20 baseline trials (perturbation = 0), 80 adaptation trials (perturbation = 1.0), 50 washout trials (perturbation = 0).

In [ ]:
# Build perturbation schedule
n_base, n_adapt, n_wash = 20, 80, 50
perturbation = np.concatenate([
    np.zeros(n_base),           # baseline: no perturbation
    np.ones(n_adapt),           # adaptation: perturbation = 1.0
    np.zeros(n_wash),           # washout: perturbation removed
])
N_total = len(perturbation)

error, xf, xs, xtotal = simulate_two_stage(perturbation)

# ── Reproduce Figure 3A–B ──
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
trials = np.arange(N_total)

# Panel A: Trial-by-trial error
ax = axes[0]
ax.plot(trials, error, 'k-', lw=1.2, label='Error')
ax.axvspan(0, n_base, alpha=0.08, color='gray', label='Baseline')
ax.axvspan(n_base, n_base+n_adapt, alpha=0.08, color='blue', label='Adaptation')
ax.axvspan(n_base+n_adapt, N_total, alpha=0.08, color='red', label='Washout')
ax.axhline(0, color='gray', lw=0.5)
ax.set_xlabel('Trial'); ax.set_ylabel('Error'); ax.set_title('A: Trial-by-trial error')
ax.legend(fontsize=8, loc='upper right')

# Panel B: Fast/slow decomposition
ax = axes[1]
ax.plot(trials, xf, 'r--', lw=1.5, label='Fast process $x_f$')
ax.plot(trials, xs, 'g-', lw=2, label='Slow process $x_s$')
ax.plot(trials, xtotal, 'k-', lw=1, alpha=0.5, label='Total $x_f + x_s$')
ax.axhline(0, color='gray', lw=0.5)
ax.set_xlabel('Trial'); ax.set_ylabel('Adaptation state'); ax.set_title('B: Fast / slow decomposition')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

print(f'Peak fast process: {xf.max():.3f} (trial {xf.argmax()})')
print(f'Peak slow process: {xs.max():.3f} (trial {xs.argmax()})')

## Part 2: Spontaneous Recovery and Savings

Two phenomena that a single-process model **cannot** explain simultaneously:

- **Spontaneous recovery:** After washout, error transiently reappears because the fast process extinguishes quickly while the slow process persists.
- **Savings:** Re-learning is faster than initial learning because $x_s$ retains from the first exposure.

### Exercise 2: Extend the schedule for re-learning (1 line)

Add 60 re-adaptation trials (perturbation = 1.0) after the washout.

In [ ]:
# Build extended schedule: baseline → adapt → washout → re-adapt
n_readapt = 60

### YOUR CODE: build the extended perturbation schedule (1 line) ###
# perturbation_ext = np.concatenate([...])

error_ext, xf_ext, xs_ext, xtotal_ext = simulate_two_stage(perturbation_ext)

# Also run a single-process model (only slow process, no fast)
error_1p, _, xs_1p, xtotal_1p = simulate_two_stage(perturbation_ext, Af=0.0, eta_f=0.0, As=0.95, eta_s=0.08)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

N_ext = len(perturbation_ext)
trials_ext = np.arange(N_ext)

# Panel A: Full learning curve (two-stage)
ax = axes[0]
ax.plot(trials_ext, error_ext, 'k-', lw=1)
ax.axvspan(n_base+n_adapt, n_base+n_adapt+n_wash, alpha=0.1, color='red')
ax.axvspan(n_base+n_adapt+n_wash, N_ext, alpha=0.1, color='green', label='Re-adapt')
ax.set_title('A: Two-stage model'); ax.set_xlabel('Trial'); ax.set_ylabel('Error')
ax.legend(fontsize=8)

# Panel B: Washout zoom — spontaneous recovery
ax = axes[1]
wash_start = n_base + n_adapt
wash_end = wash_start + n_wash
ax.plot(trials_ext[wash_start:wash_end], error_ext[wash_start:wash_end], 'k-o', ms=3, lw=1)
ax.set_title('B: Washout zoom (spontaneous recovery)'); ax.set_xlabel('Trial'); ax.set_ylabel('Error')
ax.annotate('Spontaneous\nrecovery', xy=(wash_start+15, error_ext[wash_start+15]),
            xytext=(wash_start+25, 0.3), fontsize=9,
            arrowprops=dict(arrowstyle='->', color='red'), color='red')

# Panel C: Savings — compare initial learning vs re-learning
ax = axes[2]
readapt_start = wash_end
init_errors = error_ext[n_base:n_base+30]
re_errors = error_ext[readapt_start:readapt_start+30]
ax.plot(np.arange(30), init_errors, 'b-o', ms=3, label='Initial learning')
ax.plot(np.arange(30), re_errors, 'g-o', ms=3, label='Re-learning')
ax.set_title('C: Savings'); ax.set_xlabel('Trials into adaptation'); ax.set_ylabel('Error')
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

# Compare with single-process model
print(f'Two-stage savings: initial error at trial 5 = {init_errors[4]:.3f}, re-learning = {re_errors[4]:.3f}')
print(f'Single-process: no savings possible (error resets fully during washout)')

**Q2a:** Look at Panel B. During washout, the fast process extinguishes in ~10 trials ($A_f = 0.85$, so it loses 15% per trial). What happens to the slow process during those same trials? Why does the error transiently reappear?

**Q2b:** Why can't a single-process model produce both fast initial learning *and* savings?

## Part 3: Force-Field Adaptation on the Nonlinear Arm

A velocity-dependent curl force field applies $\mathbf{F} = B_{\text{curl}} \cdot \mathbf{v}_{\text{hand}}$, where $B_{\text{curl}} = \alpha \begin{bmatrix} 0 & -1 \\ 1 & 0 \end{bmatrix}$ and $\alpha = 15$ N·s/m.

The controller uses inverse dynamics feedforward + PD feedback + a learned compensation $\hat{a}$ that is updated trial by trial via the two-stage model.

### Exercise 3: Update the fast and slow states (2 lines)

After each trial, compute the perpendicular error and update $\hat{a}$ using the two-stage model.

In [ ]:
# ── Force-field simulation infrastructure (provided) ──
from smc import mass_matrix, coriolis, rk4_step

def simulate_reach_curlfield(q0, q_target, a_hat, alpha_curl=15.0,
                              Kp=np.array([40,30]), Kd=np.array([4,3]),
                              T=0.45, dt=0.001):
    """Simulate a single reach with curl field and learned compensation.
    Returns hand path (M,2), perpendicular error at peak velocity."""
    N_steps = int(T / dt)
    q = q0.copy(); qd = np.zeros(2)
    hand_path = []
    hand_target = arm.forward_kinematics(q_target)
    
    # Minimum-jerk reference trajectory
    t_arr = np.linspace(0, T, N_steps)
    s = 10*(t_arr/T)**3 - 15*(t_arr/T)**4 + 6*(t_arr/T)**5
    sd = (30*(t_arr/T)**2 - 60*(t_arr/T)**3 + 30*(t_arr/T)**4) / T
    hand_start = arm.forward_kinematics(q0)
    ref_hand = hand_start + np.outer(s, hand_target - hand_start)
    ref_vel = np.outer(sd, hand_target - hand_start)
    
    peak_vel_idx = 0; peak_vel = 0
    
    for i in range(N_steps):
        hand = arm.forward_kinematics(q)
        J = arm.jacobian(q)
        hand_vel = J @ qd
        hand_path.append(hand.copy())
        
        speed = np.linalg.norm(hand_vel)
        if speed > peak_vel:
            peak_vel = speed; peak_vel_idx = i
        
        # Inverse dynamics feedforward for reference trajectory
        q_ref = arm.inverse_kinematics(*ref_hand[i])
        M = mass_matrix(q)
        c = coriolis(q, qd)
        
        # PD feedback
        tau_fb = Kp * (q_ref - q) - Kd * qd
        
        # Learned curl-field compensation
        B_curl = alpha_curl * np.array([[0, -1], [1, 0]])
        F_comp = a_hat * B_curl @ hand_vel
        tau_comp = J.T @ F_comp
        
        # Actual curl field perturbation
        F_curl = B_curl @ hand_vel
        tau_curl = J.T @ F_curl
        
        tau_total = tau_fb + tau_comp - tau_curl  # compensation cancels perturbation
        qdd = np.linalg.solve(M, tau_total - c)
        qd = qd + qdd * dt
        q = q + qd * dt
    
    hand_path = np.array(hand_path)
    # Perpendicular error: lateral deviation at peak velocity
    reach_dir = hand_target - hand_start
    reach_dir = reach_dir / np.linalg.norm(reach_dir)
    perp_dir = np.array([-reach_dir[1], reach_dir[0]])
    deviation = hand_path[peak_vel_idx] - hand_start
    perp_error = np.abs(np.dot(deviation, perp_dir))
    
    return hand_path, perp_error

print('Force-field simulation ready.')

In [ ]:
# ── Multi-trial adaptation ──
q0 = Q_REF.copy()
q_tgt = arm.inverse_kinematics(*(arm.forward_kinematics(Q_REF) + np.array([0.10, 0.0])))

n_trials = 80
perp_errors = np.zeros(n_trials)
a_hat = 0.0       # learned compensation (0 = no compensation)
xf_ff = 0.0; xs_ff = 0.0  # two-stage state
Af, As, eta_f, eta_s = 0.85, 0.998, 0.15, 0.03
hand_paths = {}    # save a few trajectories for plotting

for trial in range(n_trials):
    hand_path, perp_err = simulate_reach_curlfield(q0, q_tgt, a_hat)
    perp_errors[trial] = perp_err
    if trial in [0, 9, 29, 79]:
        hand_paths[trial] = hand_path.copy()
    
    e = 1.0 - a_hat  # residual un-compensated fraction
    ### YOUR CODE: update xf_ff and xs_ff (2 lines) ###
    # xf_ff = ...
    # xs_ff = ...
    a_hat = xf_ff + xs_ff

print(f'Trial 1 perp error: {perp_errors[0]*100:.1f} cm')
print(f'Trial 80 perp error: {perp_errors[-1]*100:.1f} cm')
print(f'Final compensation a_hat: {a_hat:.3f}')

In [ ]:
# ── Reproduce Figure 2 ──
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Panel A: Hand paths
ax = axes[0]
colors = {0: 'red', 9: 'orange', 29: 'green', 79: 'purple'}
labels = {0: 'Trial 1', 9: 'Trial 10', 29: 'Trial 30', 79: 'Trial 80'}
# Baseline (no field)
hp_base, _ = simulate_reach_curlfield(q0, q_tgt, a_hat=1.0)  # perfect compensation
ax.plot(hp_base[:,0]*100, hp_base[:,1]*100, 'b-', lw=2, label='Baseline', alpha=0.5)
for t_idx in [0, 9, 29, 79]:
    hp = hand_paths[t_idx]
    ax.plot(hp[:,0]*100, hp[:,1]*100, color=colors[t_idx], lw=1.5, label=labels[t_idx])
ax.set_xlabel('x (cm)'); ax.set_ylabel('y (cm)')
ax.set_title('A: Hand paths across adaptation'); ax.legend(fontsize=8)
ax.set_aspect('equal')

# Panel B: Learning curve
ax = axes[1]
ax.plot(np.arange(1, n_trials+1), perp_errors*100, 'k-o', ms=2, lw=1)
ax.set_xlabel('Trial'); ax.set_ylabel('Perpendicular error (cm)')
ax.set_title('B: Perpendicular error learning curve')
plt.tight_layout(); plt.show()

## Part 4: Fit the Two-Stage Model to Simulated Data

The force-field simulation in Part 3 generated a learning curve (perpendicular error across 80 trials). That simulation used the two-stage model with specific parameters ($A_f = 0.85$, $\eta_f = 0.15$). Now suppose you didn't know those parameters — could you recover them from the data?

We fix the slow process ($A_s = 0.998$, $\eta_s = 0.03$) and search over the fast process ($A_f$, $\eta_f$).

In [ ]:
# ── Step 1: What are we fitting? ──
# Normalise the perpendicular errors from Part 3 so trial 1 = 1.0
target_errors = perp_errors / perp_errors[0]

# ── Step 2: How well does the default model match? ──
pert_ff = np.ones(n_trials)  # perturbation = 1 on every adaptation trial
err_default, _, _, _ = simulate_two_stage(pert_ff, Af=0.85, As=0.998, eta_f=0.15, eta_s=0.03)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(np.arange(n_trials), target_errors, 'ko', ms=3, label='Observed (Part 3 simulation)')
ax.plot(np.arange(n_trials), err_default, 'b-', lw=2, label='Two-stage model (default params)')
ax.set_xlabel('Trial'); ax.set_ylabel('Normalised error')
ax.set_title('Target data vs default two-stage model'); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()
print('The default model fits well \u2014 the data was generated with these parameters.')
print('But what if we did not know Af and eta_f?')

### Exercise 4a: 1D parameter sweep (1 line)

Fix $A_f = 0.85$ and sweep $\eta_f$ from 0.01 to 0.30. For each value, simulate the two-stage model and compute the MSE against the target errors. This shows what the error landscape looks like in one dimension.

In [ ]:
etaf_vals = np.linspace(0.01, 0.30, 40)
mse_1d = np.zeros(len(etaf_vals))

for j, ef in enumerate(etaf_vals):
    ### YOUR CODE: simulate and compute MSE (1 line) ###
    # mse_1d[j] = np.mean((simulate_two_stage(pert_ff, Af=0.85, eta_f=ef, As=0.998, eta_s=0.03)[0] - target_errors)**2)
    pass

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(etaf_vals, mse_1d, 'k-o', ms=3, lw=1.5)
ax.axvline(0.15, color='blue', ls='--', lw=1, label='True $\eta_f$ = 0.15')
ax.set_xlabel('$\eta_f$ (fast learning rate)'); ax.set_ylabel('MSE')
ax.set_title('1D sweep: MSE vs $\eta_f$ (with $A_f$ fixed at 0.85)')
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()
print(f'Best eta_f: {etaf_vals[mse_1d.argmin()]:.3f} (true: 0.15)')

### Exercise 4b: 2D grid search (2 lines)

Now vary *both* $A_f$ and $\eta_f$. The 1D sweep found a sharp minimum when $A_f$ was known. The question: if we don't know $A_f$ either, is the minimum still a tight bowl — or does a valley appear where different $(A_f, \eta_f)$ pairs produce similar fits?

In [ ]:
Af_vals = np.linspace(0.5, 0.99, 30)
etaf_vals_2d = np.linspace(0.01, 0.30, 30)
mse_grid = np.zeros((len(Af_vals), len(etaf_vals_2d)))

for i, af in enumerate(Af_vals):
    for j, ef in enumerate(etaf_vals_2d):
        ### YOUR CODE: simulate and compute MSE (2 lines) ###
        # err_pred, _, _, _ = simulate_two_stage(pert_ff, Af=af, As=0.998, eta_f=ef, eta_s=0.03)
        # mse_grid[i, j] = np.mean((err_pred - target_errors)**2)
        pass

best_idx = np.unravel_index(mse_grid.argmin(), mse_grid.shape)
best_Af = Af_vals[best_idx[0]]; best_etaf = etaf_vals_2d[best_idx[1]]
print(f'Best fit: Af = {best_Af:.3f}, eta_f = {best_etaf:.3f}')
print(f'Literature values: Af = 0.85, eta_f = 0.15')

fig, ax = plt.subplots(figsize=(6, 5))
c = ax.contourf(etaf_vals_2d, Af_vals, np.log10(mse_grid + 1e-10), levels=20, cmap='viridis_r')
ax.plot(best_etaf, best_Af, 'r*', ms=15, label=f'Best fit ({best_Af:.2f}, {best_etaf:.2f})')
ax.plot(0.15, 0.85, 'w+', ms=12, mew=2, label='Smith et al. (2006)')
ax.set_xlabel('$\eta_f$ (fast learning rate)'); ax.set_ylabel('$A_f$ (fast retention)')
ax.set_title('MSE landscape (log scale)'); ax.legend(fontsize=9)
plt.colorbar(c, label='log\u2081\u2080(MSE)'); plt.tight_layout(); plt.show()

**Q4a:** Compare the 1D sweep (sharp minimum) with the 2D landscape. Is the 2D minimum a tight bowl or an elongated valley? What does this tell you about whether $A_f$ and $\eta_f$ can be identified independently from adaptation data?

**Q4b:** If you were fitting this model to *real* experimental data (noisier, fewer trials), would the identifiability problem be better or worse?

## Part 5: Cerebellar Degradation — Reducing $\eta_f$

The fast process is hypothesized to depend on the cerebellum — rapid, trial-by-trial error correction driven by climbing fiber signals. If the cerebellum is damaged, $\eta_f$ should be selectively reduced while the slow process ($\eta_s$, mediated by motor cortex) is spared.

### Exercise 5: Simulate cerebellar degeneration (1 line)

Re-run the two-stage model on the baseline → adaptation → washout schedule with $\eta_f = 0.02$ (severely reduced from 0.15).

In [ ]:
# Full schedule: baseline → adapt → washout
pert_full = np.concatenate([np.zeros(n_base), np.ones(n_adapt), np.zeros(n_wash)])

# Healthy
err_h, xf_h, xs_h, xt_h = simulate_two_stage(pert_full)

### YOUR CODE: cerebellar-degraded model (1 line) ###
# err_cb, xf_cb, xs_cb, xt_cb = simulate_two_stage(pert_full, eta_f=...)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Panel A: Error comparison
ax = axes[0]
ax.plot(err_h, 'b-', lw=1.2, label='Healthy', alpha=0.7)
ax.plot(err_cb, 'r-', lw=1.2, label='Cerebellar ($\eta_f$=0.02)', alpha=0.7)
ax.axhline(0, color='gray', lw=0.5)
ax.set_title('A: Error — Healthy vs Cerebellar'); ax.set_xlabel('Trial'); ax.set_ylabel('Error')
ax.legend(fontsize=9)

# Panel B: Fast process comparison
ax = axes[1]
ax.plot(xf_h, 'b--', lw=1.5, label='Healthy $x_f$')
ax.plot(xf_cb, 'r--', lw=1.5, label='Cerebellar $x_f$')
ax.set_title('B: Fast process'); ax.set_xlabel('Trial'); ax.set_ylabel('$x_f$')
ax.legend(fontsize=9)

# Panel C: Slow process comparison
ax = axes[2]
ax.plot(xs_h, 'b-', lw=2, label='Healthy $x_s$')
ax.plot(xs_cb, 'r-', lw=2, label='Cerebellar $x_s$')
ax.set_title('C: Slow process (preserved)'); ax.set_xlabel('Trial'); ax.set_ylabel('$x_s$')
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

adapt_end = n_base + n_adapt
print(f'Healthy error at trial 80: {err_h[adapt_end-1]:.3f}')
print(f'Cerebellar error at trial 80: {err_cb[adapt_end-1]:.3f}')

**Q5a:** The slow process (Panel C) looks nearly identical between healthy and cerebellar-degraded models. Why?

**Q5b:** Smith & Shadmehr (2005) found that cerebellar patients' motor commands *do* change trial by trial, but the changes are *random* — uninformed by the preceding error. How does reducing $\eta_f$ in the model capture this finding? What aspect of the clinical picture does it *not* capture?

## Part 6: REINFORCE for Visuomotor Rotation

In a visuomotor rotation (VMR) task with **reward-only feedback**, the subject reaches in the dark and receives only a scalar reward signal (no cursor, no directional information). The REINFORCE update rule converts this scalar feedback into a directional update:

$$\Delta\theta = \eta \cdot \delta \cdot \varepsilon$$

where $\varepsilon \sim \mathcal{N}(0, \sigma^2)$ is exploration noise added to the reach direction, $\delta = r - \bar{r}$ is the reward prediction error (reward minus running average), and $\eta$ is the learning rate.

**A note on parameters:** The REINFORCE parameters below ($\sigma = 6°$, $\eta = 0.15$, Gaussian reward width $\sigma_r = 10°$) are chosen to produce the qualitative dissociation reported by Izawa & Shadmehr (2011). They are *not* fit to their experimental data. The specific learning curves are predictions of this model.

### Exercise 6a: Implement REINFORCE (3 lines)

Complete the three missing lines: sample exploration noise, compute reward, compute the update.

In [ ]:
def simulate_reinforce(rotation, n_trials=200, sigma=6.0, eta=0.15, target_zone=10.0,
                       rng=None):
    """Simulate reward-based VMR adaptation using REINFORCE.
    
    Parameters
    ----------
    rotation : float — rotation angle in degrees (e.g. 30)
    sigma : float — exploration noise SD (degrees)
    eta : float — learning rate
    target_zone : float — reward width (degrees); Gaussian reward = exp(-error²/2σ²_reward)
    
    Returns
    -------
    theta_hist : (n_trials,) — compensation angle at each trial
    reward_hist : (n_trials,) — reward received at each trial
    """
    if rng is None: rng = np.random.default_rng()
    theta = 0.0  # current compensation angle
    r_bar = 0.0  # running average reward
    theta_hist = np.zeros(n_trials)
    reward_hist = np.zeros(n_trials)
    
    for n in range(n_trials):
        ### YOUR CODE: sample exploration noise (1 line) ###
        # epsilon = rng.normal(0, sigma)
        
        # Actual reach direction = compensation + exploration
        reach_dir = theta + epsilon
        # Cursor direction (what hits the screen) = reach + rotation
        cursor_error = rotation - reach_dir  # degrees from target
        
        ### YOUR CODE: compute Gaussian reward (1 line) ###
        # reward = np.exp(-cursor_error**2 / (2 * target_zone**2))
        
        # RPE
        delta = reward - r_bar
        r_bar = 0.9 * r_bar + 0.1 * reward  # exponential moving average
        
        ### YOUR CODE: REINFORCE update (1 line) ###
        # theta = theta + eta * delta * epsilon
        
        theta_hist[n] = theta
        reward_hist[n] = reward
    
    return theta_hist, reward_hist

print('REINFORCE function ready.')

### Exercise 6b: Run the comparison — error-based vs reward-based (1 line)

Run the error-based model (two-stage on a 30° rotation) and the reward-based model (REINFORCE, averaged over 30 random seeds).

In [ ]:
rotation = 30.0  # degrees
n_adapt_vmr = 200
n_baseline_vmr = 10
n_seeds = 30

# Error-based: two-stage model on VMR
pert_vmr = np.concatenate([np.zeros(n_baseline_vmr), np.full(n_adapt_vmr, rotation)])
err_vmr, _, _, adapt_vmr = simulate_two_stage(pert_vmr / rotation)  # normalise to [0,1]
compensation_error = adapt_vmr * rotation  # scale back to degrees
cursor_error_eb = rotation - compensation_error  # what error-based learner sees

# Reward-based: REINFORCE averaged over seeds
### YOUR CODE: run REINFORCE for n_seeds seeds, collect theta_hist arrays (1 line or small loop) ###
theta_all = np.zeros((n_seeds, n_adapt_vmr))
reward_all = np.zeros((n_seeds, n_adapt_vmr))
for s in range(n_seeds):
    theta_all[s], reward_all[s] = simulate_reinforce(rotation, n_adapt_vmr, rng=np.random.default_rng(s))

theta_mean = theta_all.mean(axis=0)
theta_std = theta_all.std(axis=0)
reward_mean = reward_all.mean(axis=0)

# ── Reproduce Figure 5B ──
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Panel A: Cursor error (learning curves)
ax = axes[0]
trials_eb = np.arange(n_baseline_vmr, n_baseline_vmr + n_adapt_vmr)
trials_rl = np.arange(n_adapt_vmr)
ax.plot(trials_eb, cursor_error_eb[n_baseline_vmr:], 'b-', lw=2, label='Error-based (cursor on)')
ax.plot(trials_rl, rotation - theta_mean, 'r-', lw=2, label='Reward-based (cursor off)')
ax.fill_between(trials_rl, rotation - theta_mean - theta_std, rotation - theta_mean + theta_std,
                color='red', alpha=0.15)
ax.axhline(0, color='gray', lw=0.5, ls='--')
ax.set_xlabel('Trial'); ax.set_ylabel('Cursor error (°)')
ax.set_title('A: Learning curves'); ax.legend(fontsize=9)

# Panel B: Compensation angle
ax = axes[1]
ax.plot(trials_eb, compensation_error[n_baseline_vmr:], 'b-', lw=2, label='Error-based')
for s in range(min(5, n_seeds)):
    ax.plot(trials_rl, theta_all[s], 'r-', alpha=0.1, lw=0.5)
ax.plot(trials_rl, theta_mean, 'r-', lw=2, label=f'Reward-based (mean, n={n_seeds})')
ax.axhline(rotation, color='gray', lw=0.5, ls='--', label='Full compensation (30°)')
ax.set_xlabel('Trial'); ax.set_ylabel('Compensation angle (°)')
ax.set_title('B: Compensation angle'); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

print(f'Error-based: final compensation = {compensation_error[-1]:.1f}°')
print(f'Reward-based: mean final compensation = {theta_mean[-1]:.1f}° ± {theta_std[-1]:.1f}°')

## Part 7: The Exploration–Exploitation Tradeoff

The exploration noise $\sigma$ controls how widely the RL learner explores. Too little exploration → the learner never finds the target. Too much → rewards are too noisy to learn from.

### Exercise 7: Vary $\sigma$ and measure asymptotic compensation (1 line)

Run REINFORCE for $\sigma \in \{2, 4, 6, 8, 10, 12\}$ degrees and record the mean compensation over the last 50 trials.

In [ ]:
sigma_vals = [2, 4, 6, 8, 10, 12]
asymptotic_comp = []
learning_speed = []  # trials to reach 50% of final compensation

for sigma in sigma_vals:
    theta_runs = np.zeros((n_seeds, n_adapt_vmr))
    for s in range(n_seeds):
        theta_runs[s], _ = simulate_reinforce(rotation, n_adapt_vmr, sigma=sigma,
                                               rng=np.random.default_rng(s))
    mean_theta = theta_runs.mean(axis=0)
    
    ### YOUR CODE: compute asymptotic compensation (last 50 trials mean) (1 line) ###
    # asymp = mean_theta[-50:].mean()
    
    asymptotic_comp.append(asymp)
    # Learning speed: first trial where mean > asymp/2
    half_idx = np.where(mean_theta > asymp/2)[0]
    learning_speed.append(half_idx[0] if len(half_idx) > 0 else n_adapt_vmr)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ax = axes[0]
ax.plot(sigma_vals, asymptotic_comp, 'ko-', lw=2)
ax.axhline(rotation, color='gray', ls='--', lw=0.5, label='Full compensation')
ax.set_xlabel('Exploration noise σ (°)'); ax.set_ylabel('Asymptotic compensation (°)')
ax.set_title('A: Compensation vs exploration'); ax.legend(fontsize=9)

ax = axes[1]
ax.plot(sigma_vals, learning_speed, 'ko-', lw=2)
ax.set_xlabel('Exploration noise σ (°)'); ax.set_ylabel('Trials to 50% compensation')
ax.set_title('B: Learning speed vs exploration')
plt.tight_layout(); plt.show()

**Q7a:** Why does asymptotic compensation *increase* with σ (up to a point) even though larger σ also makes individual trials noisier?

**Q7b:** Izawa & Shadmehr (2011) found that reward-based learning did *not* recalibrate proprioception, unlike error-based learning. In the context of this simulation, why would REINFORCE not update the forward model (A, B) — what information is missing from the reward signal?

## Part 8: Two Systems, Two Failure Modes (Synthesis)

The OFC framework has two learned components: the forward model (A, B → cerebellum) and the cost function (Q, R → basal ganglia). Damage to each produces a distinct clinical picture.

No new code — the cell below runs both learning systems under three conditions: healthy, cerebellar damage ($\eta_f \to 0.02$), and basal ganglia damage (RL exploration $\to 0$).

In [ ]:
# ── Part 8: Two rows, three columns ──
# Top row: Error-based adaptation (two-stage model)
# Bottom row: Reward-based learning (REINFORCE, 30-seed average)

n_seeds_p8 = 30

def avg_reinforce_p8(n_trials, **kwargs):
    theta_runs = np.zeros((n_seeds_p8, n_trials))
    for s in range(n_seeds_p8):
        theta_runs[s], _ = simulate_reinforce(rotation, n_trials, rng=np.random.default_rng(s), **kwargs)
    return theta_runs.mean(axis=0)

fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharex='row')

# ── Error-based row (top) ──
# Healthy
err_h, _, _, _ = simulate_two_stage(pert_full)
axes[0,0].plot(err_h, 'b-', lw=1.5)
axes[0,0].set_title('A: Healthy'); axes[0,0].set_ylabel('Adaptation error')
axes[0,0].axhline(0, color='gray', lw=0.5)

# Cerebellar damage: eta_f degraded
err_cb, _, _, _ = simulate_two_stage(pert_full, eta_f=0.02)
axes[0,1].plot(err_cb, 'b-', lw=1.5)
axes[0,1].set_title('B: Cerebellar damage')
axes[0,1].axhline(0, color='gray', lw=0.5)

# BG damage: error-based intact (same as healthy)
err_bg, _, _, _ = simulate_two_stage(pert_full)
axes[0,2].plot(err_bg, 'b-', lw=1.5)
axes[0,2].set_title('C: Basal ganglia damage')
axes[0,2].axhline(0, color='gray', lw=0.5)

# Set same y-limits for top row
ylim_top = (min(err_h.min(), err_cb.min()) - 0.1, max(err_h.max(), err_cb.max()) + 0.1)
for ax in axes[0,:]:
    ax.set_ylim(ylim_top)

# Label the row

# ── Reward-based row (bottom) ──
# Healthy RL
theta_h = avg_reinforce_p8(n_adapt_vmr)
axes[1,0].plot(np.arange(n_adapt_vmr), theta_h, 'r-', lw=1.5)
axes[1,0].set_ylabel('RL compensation (°)')
axes[1,0].set_xlabel('Trial')
axes[1,0].axhline(rotation, color='gray', lw=0.5, ls='--')

# Cerebellar damage: RL intact (same as healthy)
theta_cb = avg_reinforce_p8(n_adapt_vmr)
axes[1,1].plot(np.arange(n_adapt_vmr), theta_cb, 'r-', lw=1.5)
axes[1,1].set_xlabel('Trial')
axes[1,1].axhline(rotation, color='gray', lw=0.5, ls='--')

# BG damage: RL impaired
theta_bg = avg_reinforce_p8(n_adapt_vmr, sigma=0.5, eta=0.01)
axes[1,2].plot(np.arange(n_adapt_vmr), theta_bg, 'r-', lw=1.5)
axes[1,2].set_xlabel('Trial')
axes[1,2].axhline(rotation, color='gray', lw=0.5, ls='--')

# Set same y-limits for bottom row
ylim_bot = (-2, max(theta_h.max(), rotation) + 2)
for ax in axes[1,:]:
    ax.set_ylim(ylim_bot)

# Label the row

# Annotate impaired panels
axes[0,1].text(0.95, 0.95, 'IMPAIRED', transform=axes[0,1].transAxes,
               fontsize=11, fontweight='bold', color='red', ha='right', va='top',
               bbox=dict(boxstyle='round,pad=0.3', facecolor='mistyrose', edgecolor='red', alpha=0.8))
axes[1,2].text(0.95, 0.95, 'IMPAIRED', transform=axes[1,2].transAxes,
               fontsize=11, fontweight='bold', color='red', ha='right', va='top',
               bbox=dict(boxstyle='round,pad=0.3', facecolor='mistyrose', edgecolor='red', alpha=0.8))

# Row labels
axes[0,0].set_ylabel('Adaptation error\n(error-based)', fontsize=10)
axes[1,0].set_ylabel('RL compensation \u00b0\n(reward-based)', fontsize=10)
plt.tight_layout(w_pad=3)
plt.show()

print('Top row: error-based adaptation (two-stage model on force-field schedule)')
print('Bottom row: reward-based learning (REINFORCE on 30° VMR, 30-seed average)')
print()
print('Cerebellar damage (column B): error-based IMPAIRED, reward-based PRESERVED')
print('BG damage (column C): error-based PRESERVED, reward-based IMPAIRED')


### Discussion Questions

**Q8a:** Map each failure mode back to the OFC framework. For cerebellar damage, which OFC component breaks (A/B or Q/R), and what clinical symptom does it predict? For basal ganglia damage, which component breaks, and what symptom does it predict?

**Q8b:** Both cerebellar and Parkinson's patients can improve with practice. Based on the two-stage model, which *process* would cerebellar patients rely on for improvement? Based on Mazzoni et al. (2007), what kind of intervention helps Parkinson's patients bypass the basal ganglia's action selection?

**Q8c:** Consider a patient who shows *both* impaired error-based adaptation and impaired reward-based learning. What would you predict about their motor behavior? Which brain structures might be affected?

## Summary

| Part | What you built | Lecture figure reproduced |
|------|---------------|--------------------------|
| 1 | Two-stage state-space model | Fig 3A–B |
| 2 | Spontaneous recovery & savings | Fig 3C |
| 3 | Force-field adaptation on arm | Fig 2B |
| 4 | Parameter fitting (grid search) | — (new) |
| 5 | Cerebellar degradation ($\eta_f$) | — (new) |
| 6 | REINFORCE for VMR | Fig 5B |
| 7 | Exploration–exploitation tradeoff | — (new) |
| 8 | Two systems, two failure modes | — (synthesis) |

**Key takeaway:** The OFC framework has two learned components. The cerebellum learns the forward model (A, B) via error-based learning — the two-stage model captures the trial-by-trial dynamics. The basal ganglia shape the cost function (Q, R) via reinforcement learning — REINFORCE shows how a scalar reward signal can drive adaptation, albeit slowly and noisily. Damage to either system produces distinct, predictable motor deficits.